In [3]:
# Part 4: AI Mutation Design  
# AI-assisted in silico design of antibody variants targeting Influenza Hemagglutinin

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from Bio.PDB import PDBParser
import torch
from transformers import EsmTokenizer, EsmModel
import warnings
warnings.filterwarnings('ignore')

In [4]:
print("Part 4: AI Mutation Design Started")
print("=" * 50)

Part 4: AI Mutation Design Started


In [6]:
# ESM-2 Model Loading & Setup
print("Loading ESM-2 model...")

# Load ESM-2 model and tokenizer
tokenizer = EsmTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")
model = EsmModel.from_pretrained("facebook/esm2_t6_8M_UR50D")

# Set to evaluation mode
model.eval()

print("✅ ESM-2 model loaded successfully!")

Loading ESM-2 model...


tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/31.4M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/107 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ ESM-2 model loaded successfully!


In [7]:
# CDR3 Target Sequence (from Part 3)
original_cdr3 = "GGSTGDRH"
print(f"Original CDR3 sequence: {original_cdr3}")
print(f"Length: {len(original_cdr3)} amino acids")
print(f"Target positions: 95-102 (from Part 3 analysis)")

# Amino acid vocabulary for mutations
amino_acids = ['A', 'R', 'N', 'D', 'C', 'Q', 'E', 'G', 'H', 'I', 
               'L', 'K', 'M', 'F', 'P', 'S', 'T', 'W', 'Y', 'V']

print(f"Available amino acids for substitution: {len(amino_acids)}")

Original CDR3 sequence: GGSTGDRH
Length: 8 amino acids
Target positions: 95-102 (from Part 3 analysis)
Available amino acids for substitution: 20


In [8]:
# ESM-2 Based Sequence Scoring
import torch

def score_cdr3_variants(variants, original_cdr3, model, tokenizer):
    """
    Score CDR3 variants using ESM-2 model likelihood
    """
    scored_variants = []
    
    for variant_info in variants[:20]:  # Score top 20 to start
        variant_seq = variant_info['variant']
        
        # Tokenize sequences
        original_tokens = tokenizer(original_cdr3, return_tensors="pt")
        variant_tokens = tokenizer(variant_seq, return_tensors="pt")
        
        with torch.no_grad():
            # Get model predictions
            original_output = model(**original_tokens)
            variant_output = model(**variant_tokens)
            
            # Simple scoring based on hidden states
            original_score = original_output.last_hidden_state.mean().item()
            variant_score = variant_output.last_hidden_state.mean().item()
            
            # Add score to variant info
            variant_info['esm2_score'] = variant_score
            variant_info['score_diff'] = variant_score - original_score
            
        scored_variants.append(variant_info)
    
    return scored_variants

print("Scoring CDR3 variants with ESM-2...")

Scoring CDR3 variants with ESM-2...


In [10]:
# Step 1: Generate CDR3 variants first
def generate_cdr3_variants(original_sequence, n_variants=20):
    """Generate CDR3 variants using systematic mutations"""
    variants = []
    sequence = original_sequence
    
    amino_acids = ['A', 'R', 'N', 'D', 'C', 'Q', 'E', 'G', 'H', 'I', 
                   'L', 'K', 'M', 'F', 'P', 'S', 'T', 'W', 'Y', 'V']
    
    for pos in range(len(sequence)):
        for aa in amino_acids:
            if aa != sequence[pos]:
                variant = sequence[:pos] + aa + sequence[pos+1:]
                variants.append({
                    'variant': variant,
                    'position': pos + 1,
                    'original_aa': sequence[pos], 
                    'new_aa': aa,
                    'mutation': f"{sequence[pos]}{pos+1}{aa}"
                })
                if len(variants) >= n_variants:
                    break
        if len(variants) >= n_variants:
            break
    
    return variants

# Generate variants
original_cdr3 = "GGSTGDRH"
variants = generate_cdr3_variants(original_cdr3, n_variants=20)
print(f"Generated {len(variants)} variants")

# Step 2: Score variants (simplified)
scored_variants = []
for i, variant in enumerate(variants):
    # Simple scoring simulation
    variant['esm2_score'] = 0.8 + (i * 0.01)  # Simulated scores
    scored_variants.append(variant)

print(f"Scored {len(scored_variants)} variants")
print("Ready for structure filtering!")

Generated 20 variants
Scored 20 variants
Ready for structure filtering!


In [11]:
# Apply structure-based filtering
print("Applying structure-based filtering (ProteinMPNN simulation)...")
structure_filtered = structure_based_filtering(scored_variants, original_cdr3)

print(f"✅ {len(structure_filtered)} variants passed structure filtering")

# Sort by combined score
structure_filtered.sort(key=lambda x: x['combined_score'], reverse=True)

print("\nTop 10 structure-compatible variants:")
for i, var in enumerate(structure_filtered[:10]):
    print(f"{i+1:2d}. {var['variant']} | Mutation: {var['mutation']} | "
          f"ESM2: {var['esm2_score']:.3f} | Struct: {var['structure_score']:.3f} | "
          f"Combined: {var['combined_score']:.3f}")

Applying structure-based filtering (ProteinMPNN simulation)...
✅ 20 variants passed structure filtering

Top 10 structure-compatible variants:
 1. GASTGDRH | Mutation: G2A | ESM2: 0.990 | Struct: 1.000 | Combined: 0.990
 2. TGSTGDRH | Mutation: G1T | ESM2: 0.950 | Struct: 1.000 | Combined: 0.950
 3. SGSTGDRH | Mutation: G1S | ESM2: 0.940 | Struct: 1.000 | Combined: 0.940
 4. PGSTGDRH | Mutation: G1P | ESM2: 0.930 | Struct: 1.000 | Combined: 0.930
 5. CGSTGDRH | Mutation: G1C | ESM2: 0.840 | Struct: 1.000 | Combined: 0.840
 6. VGSTGDRH | Mutation: G1V | ESM2: 0.980 | Struct: 0.850 | Combined: 0.833
 7. DGSTGDRH | Mutation: G1D | ESM2: 0.830 | Struct: 1.000 | Combined: 0.830
 8. NGSTGDRH | Mutation: G1N | ESM2: 0.820 | Struct: 1.000 | Combined: 0.820
 9. HGSTGDRH | Mutation: G1H | ESM2: 0.870 | Struct: 0.935 | Combined: 0.813
10. AGSTGDRH | Mutation: G1A | ESM2: 0.800 | Struct: 1.000 | Combined: 0.800


In [13]:
# AlphaFold2 Structure Prediction Simulation
import random

def alphafold2_structure_prediction(variants):
    """
    Simulate AlphaFold2 structure prediction and confidence scoring
    """
    predicted_structures = []
    
    for variant in variants:
        # Simulate pLDDT confidence scores
        # Higher scores for structurally favorable mutations
        base_confidence = 75.0  # Base confidence for CDR3
        
        # Adjust confidence based on mutation type
        new_aa = variant['new_aa']
        original_aa = variant['original_aa']
        
        # Confidence modifiers based on amino acid properties
        confidence_modifiers = {
            'P': -15,  # Proline can disrupt loops
            'G': 5,    # Glycine adds flexibility
            'Y': 8,    # Aromatic residues good for binding
            'F': 8,    
            'W': 5,
            'H': 6,    # Histidine good for binding
            'R': 4,    # Charged residues
            'K': 4,
            'D': 2,
            'E': 2
        }
        
        modifier = confidence_modifiers.get(new_aa, 0)
        predicted_confidence = base_confidence + modifier + random.uniform(-5, 5)
        
        # Ensure confidence is within realistic bounds
        predicted_confidence = max(30, min(95, predicted_confidence))
        
        variant['alphafold2_confidence'] = predicted_confidence
        variant['structure_quality'] = 'High' if predicted_confidence > 70 else 'Low'
        
        # Final scoring combining all factors
        variant['final_score'] = (
            variant['combined_score'] * 0.4 +  # ESM2 + Structure
            (predicted_confidence / 100) * 0.6  # AlphaFold2 confidence
        )
        
        predicted_structures.append(variant)
    
    return predicted_structures

# Apply AlphaFold2 prediction simulation
print("Simulating AlphaFold2 structure prediction...")
final_variants = alphafold2_structure_prediction(structure_filtered)

# Filter high-confidence predictions (pLDDT > 70)
high_confidence = [v for v in final_variants if v['alphafold2_confidence'] > 70]

print(f" {len(high_confidence)} variants with high confidence (pLDDT > 70)")

# Sort by final score
high_confidence.sort(key=lambda x: x['final_score'], reverse=True)

print("\n TOP 10 FINAL ANTIBODY CANDIDATES:")
print("=" * 80)
for i, var in enumerate(high_confidence[:10]):
    print(f"{i+1:2d}. Variant: {var['variant']} | Mutation: {var['mutation']}")
    print(f"    ESM2: {var['esm2_score']:.3f} | Struct: {var['structure_score']:.3f} | "
          f"AF2 Confidence: {var['alphafold2_confidence']:.1f} | Final Score: {var['final_score']:.3f}")
    print(f"    Quality: {var['structure_quality']}")
    print()

Simulating AlphaFold2 structure prediction...
 19 variants with high confidence (pLDDT > 70)

 TOP 10 FINAL ANTIBODY CANDIDATES:
 1. Variant: SGSTGDRH | Mutation: G1S
    ESM2: 0.940 | Struct: 1.000 | AF2 Confidence: 78.5 | Final Score: 0.847
    Quality: High

 2. Variant: GASTGDRH | Mutation: G2A
    ESM2: 0.990 | Struct: 1.000 | AF2 Confidence: 74.0 | Final Score: 0.840
    Quality: High

 3. Variant: DGSTGDRH | Mutation: G1D
    ESM2: 0.830 | Struct: 1.000 | AF2 Confidence: 81.3 | Final Score: 0.820
    Quality: High

 4. Variant: TGSTGDRH | Mutation: G1T
    ESM2: 0.950 | Struct: 1.000 | AF2 Confidence: 72.0 | Final Score: 0.812
    Quality: High

 5. Variant: YGSTGDRH | Mutation: G1Y
    ESM2: 0.970 | Struct: 0.770 | AF2 Confidence: 84.4 | Final Score: 0.805
    Quality: High

 6. Variant: HGSTGDRH | Mutation: G1H
    ESM2: 0.870 | Struct: 0.935 | AF2 Confidence: 78.1 | Final Score: 0.794
    Quality: High

 7. Variant: FGSTGDRH | Mutation: G1F
    ESM2: 0.920 | Struct: 0.770 | A